# 01 — Data Exploration

Explore the Avazu CTR dataset and the derived ad monetization metrics.

**Dataset:** [Avazu Click-Through Rate Prediction (Kaggle)](https://www.kaggle.com/c/avazu-ctr-prediction/data)  
**Metrics derived:** eCPM, fill_rate, CTR, impressions, ARPDAU

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Load processed hourly metrics
df = pd.read_csv('../data/processed/ad_metrics_hourly.csv', parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)
print(f'Shape: {df.shape}')
print(f'Date range: {df["datetime"].min()} → {df["datetime"].max()}')
df.head()

## Metric Distributions

In [ ]:
metrics = ['ecpm', 'fill_rate', 'ctr', 'impressions', 'arpdau']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, m in enumerate(metrics):
    axes[i].hist(df[m], bins=50, color='steelblue', alpha=0.8, edgecolor='white')
    axes[i].set_title(m.upper(), fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

axes[-1].set_visible(False)
plt.suptitle('Ad Metric Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Time Series Overview

In [ ]:
fig, axes = plt.subplots(len(metrics), 1, figsize=(16, 14), sharex=True)

for ax, m in zip(axes, metrics):
    ax.plot(df['datetime'], df[m], linewidth=0.8, color='#2c7bb6')
    ax.set_ylabel(m.upper(), fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_title('Ad Metrics Time Series (Hourly)', fontsize=13, fontweight='bold')
plt.xlabel('Datetime')
plt.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[metrics].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Metric Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey insight: Fill rate and eCPM are positively correlated — when fill rate drops, eCPM tends to fall too.')

## Hourly Patterns (Average by Hour of Day)

In [ ]:
df['hour_of_day'] = df['datetime'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly_ecpm = df.groupby('hour_of_day')['ecpm'].mean()
axes[0].bar(hourly_ecpm.index, hourly_ecpm.values, color='#4c72b0', alpha=0.85)
axes[0].set_title('Avg eCPM by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('eCPM ($)')

hourly_fill = df.groupby('hour_of_day')['fill_rate'].mean()
axes[1].bar(hourly_fill.index, hourly_fill.values, color='#dd8452', alpha=0.85)
axes[1].set_title('Avg Fill Rate by Hour of Day', fontweight='bold')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Fill Rate')

plt.tight_layout()
plt.show()

In [ ]:
print(df[metrics].describe().round(4))